# Laboratory 02 — Equations of state

In this laboratory you will build the van der Waals correction to the ideal gas law out of
two physical effects — molecular attraction and excluded volume — and use it to find a
real gas's critical point.

Work through it in order. Where the notebook asks you to predict, write your prediction in
the cell provided **before** running the next cell. That is not a ritual: a prediction you
have committed to is the only reliable way to discover that you were wrong.

## Model specification

| | |
|---|---|
| **System** | $N$ structureless molecules of a real gas, in the van der Waals mean-field approximation; the ideal gas is the special case $a=b=0$ |
| **Dynamics** | none — a static equation of state, not a trajectory to integrate |
| **Boundary** | a rigid container of volume $V$ at temperature $T$, or a piston varying $V$ along an isotherm |
| **Ensemble** | implicit thermodynamic equilibrium — $P(N,V,T)$ is an equilibrium equation of state |
| **Ignored** | higher-order interactions, quantum effects, internal molecular structure, mixtures |
| **Valid when** | dilute to moderately dense, classical, away from the critical point |
| **Failure modes** | very high density, at or below the critical temperature |

All the physics lives in `thermolab.equations_of_state` — open it and read it. Nothing in
this course is hidden inside a framework.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import equations_of_state, paths
from thermolab.constants import K_B
from thermolab.validation import relative_error

# Argon's van der Waals constants, converted to this course's per-particle convention
# (a_molar / N_A**2, b_molar / N_A -- see the module page's "A note on units" box).
ARGON_A = 3.736e-49  # Pa m^6, per molecule
ARGON_B = 5.317e-29  # m^3, per molecule

print(f"k_B = {K_B:.6e} J/K")
print(f"argon a = {ARGON_A:.4e} Pa m^6")
print(f"argon b = {ARGON_B:.4e} m^3")

## Part 1 — The ideal-gas isotherms

Start with $a=b=0$: three isotherms of the same $N$, at three temperatures. This is the flat
surface the rest of the module bends.

In [ ]:
n_particles = 5.0e22
volumes = np.linspace(1.5e-3, 6.0e-3, 200)

fig, ax = plt.subplots(figsize=(6, 4.5))
for temperature in (200.0, 300.0, 400.0):
    pressure = equations_of_state.van_der_waals_pressure(
        n_particles, temperature, volumes, 0.0, 0.0
    )
    ax.plot(volumes * 1e3, pressure / 1e3, label=f"T = {temperature:.0f} K")
ax.set_xlabel("V (L)")
ax.set_ylabel("P (kPa)")
ax.set_title("Ideal-gas isotherms (a = b = 0)")
ax.legend()
plt.tight_layout()
plt.show()

### Predict

Before running the next cell: at $N = 5.0\times10^{22}$, $V = 2.0\ \mathrm{L}$,
$T = 300\ \mathrm{K}$, will turning on argon's $a$ and $b$ raise or lower the pressure
compared to the ideal value? By roughly how much, as a percentage?

**Your prediction:**

*(write here before running the next cell)*

## Part 2 — Turning on the van der Waals correction

Same $N$, $V$, $T$ as module 4's Problem 2 — now with argon's real constants.

In [ ]:
n_particles = 5.0e22
volume = 2.0e-3
temperature = 300.0

ideal = paths.ideal_gas_pressure(n_particles, temperature, volume)
vdw = equations_of_state.van_der_waals_pressure(
    n_particles, temperature, volume, ARGON_A, ARGON_B
)
shift = (vdw - ideal) / ideal

print(f"ideal pressure  {ideal:.6e} Pa")
print(f"vdW pressure    {vdw:.6e} Pa")
print(f"relative shift  {shift:+.4%}")

volumes = np.linspace(1.5e-3, 6.0e-3, 200)
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(
    volumes * 1e3,
    paths.isothermal_pressure(volumes, n_particles, temperature) / 1e3,
    "--", color="0.5", label="ideal",
)
ax.plot(
    volumes * 1e3,
    equations_of_state.van_der_waals_pressure(
        n_particles, temperature, volumes, ARGON_A, ARGON_B
    ) / 1e3,
    color="#2563eb", label="van der Waals (argon)",
)
ax.set_xlabel("V (L)")
ax.set_ylabel("P (kPa)")
ax.set_title(f"T = {temperature:.0f} K: ideal vs argon")
ax.legend()
plt.tight_layout()
plt.show()

At this density the two corrections nearly cancel, and the net shift is small — but not
zero. Compare the sign and size of your prediction above with the printed
`relative shift`.

## Part 3 — Intensive versus extensive

Pressure has to be intensive for "the pressure of the gas" to mean anything: scaling $N$ and
$V$ together at fixed $T$ — the same substance, more of it — must leave $P$ unchanged. The
module page proves this algebraically; here we check it by direct computation.

In [ ]:
n_particles = 5.0e22
volume = 2.0e-3
temperature = 300.0

base = equations_of_state.van_der_waals_pressure(
    n_particles, temperature, volume, ARGON_A, ARGON_B
)
print(f"base:      N = {n_particles:.3e}  V = {volume:.3e} m^3  ->  P = {base:.10e} Pa\n")

for factor in (2, 5, 50):
    scaled = equations_of_state.van_der_waals_pressure(
        factor * n_particles, temperature, factor * volume, ARGON_A, ARGON_B
    )
    print(
        f"factor {factor:3d}:  P = {scaled:.10e} Pa   "
        f"relative difference from base = {relative_error(scaled, base):.2e}"
    )

## Part 4 — The critical point

`critical_point` gives $(T_c, P_c, V_c)$ in closed form. We check it two ways: against
argon's measured critical point, and against the pressure formula itself, evaluated near
$V_c$.

In [ ]:
n_particles = 6.022e23  # one mole of argon
t_c, p_c, v_c = equations_of_state.critical_point(n_particles, ARGON_A, ARGON_B)

print(f"predicted T_c = {t_c:.1f} K     (measured: 150.9 K)")
print(f"predicted P_c = {p_c:.4e} Pa   (measured: 4.87e6 Pa)")
print(f"predicted V_c = {v_c * 1e6:.2f} cm^3   (measured molar volume: ~74.6 cm^3)")
print()

for delta_fraction in (0.10, 0.01, 0.001):
    delta = delta_fraction * v_c
    p_plus = equations_of_state.van_der_waals_pressure(
        n_particles, t_c, v_c + delta, ARGON_A, ARGON_B
    )
    p_minus = equations_of_state.van_der_waals_pressure(
        n_particles, t_c, v_c - delta, ARGON_A, ARGON_B
    )
    print(
        f"delta = {delta_fraction:>5.1%} of V_c:  "
        f"P(V_c+d) = {p_plus:.6e}   P(V_c-d) = {p_minus:.6e}   (P_c = {p_c:.6e})"
    )

$T_c$ and $P_c$ land within about a percent of the measured values. $V_c$ does not — see
the module page's discussion of $P_cV_c/(Nk_BT_c)$ for why that particular comparison is the
least trustworthy one a van der Waals fit can make.

## Part 5 — Automated checks

A simulation — or a closed-form model — you have not checked is a picture, not evidence.
These are the same assertions that run in the project's test suite.

In [ ]:
# 1. The ideal-gas limit: a = b = 0 must match paths.ideal_gas_pressure exactly.
ideal_ref = paths.ideal_gas_pressure(1000, 300.0, 1e-3)
vdw_zero = equations_of_state.van_der_waals_pressure(1000, 300.0, 1e-3, 0.0, 0.0)
assert relative_error(vdw_zero, ideal_ref) < 1e-12

# 2. Intensivity: pressure is unchanged under simultaneous N, V scaling at fixed T.
# volume is well above the excluded volume N*b = 5.317e-26 m^3.
base_p = equations_of_state.van_der_waals_pressure(1000, 300.0, 1.0e-24, ARGON_A, ARGON_B)
scaled_p = equations_of_state.van_der_waals_pressure(50_000, 300.0, 5.0e-23, ARGON_A, ARGON_B)
assert relative_error(scaled_p, base_p) < 1e-10

# 3. The closed-form critical point matches the pressure formula evaluated there.
t_c, p_c, v_c = equations_of_state.critical_point(1000, ARGON_A, ARGON_B)
p_at_critical = equations_of_state.van_der_waals_pressure(1000, t_c, v_c, ARGON_A, ARGON_B)
assert relative_error(p_at_critical, p_c) < 1e-10


# 4. dP/dV vanishes at V_c: a central finite difference must shrink at second order.
def central_difference(n):
    h = v_c / n
    p_plus = equations_of_state.van_der_waals_pressure(1000, t_c, v_c + h, ARGON_A, ARGON_B)
    p_minus = equations_of_state.van_der_waals_pressure(1000, t_c, v_c - h, ARGON_A, ARGON_B)
    return (p_plus - p_minus) / (2 * h)


derivative_estimates = [abs(central_difference(n)) for n in (100, 1000)]
assert derivative_estimates[1] < derivative_estimates[0] / 50  # shrinks roughly as h^2

print("all checks passed")

## Part 6 — Explore it yourself

The sliders below let you dial $a$ and $b$ as fractions of argon's own values, and vary $N$
and $T$. Two experiments worth doing:

1. Set both fractions to $0$ and confirm the van der Waals curve lands exactly on the ideal
   one.
2. Raise the temperature slider while $a$, $b$ stay at argon's values, and watch the curve
   straighten out — this is the isotherm rising above $T_c$.

In [ ]:
import ipywidgets as widgets


def explore(n_e22=5.0, temperature=300.0, a_fraction=1.0, b_fraction=1.0):
    n_particles = n_e22 * 1e22
    a = ARGON_A * a_fraction
    b = ARGON_B * b_fraction
    v_min = 1.05 * n_particles * b if b > 0 else 1.0e-4
    volumes = np.linspace(v_min, 8.0e-3, 300)

    fig, ax = plt.subplots(figsize=(6.5, 4.5))
    ax.plot(
        volumes * 1e3,
        paths.isothermal_pressure(volumes, n_particles, temperature) / 1e3,
        "--", color="0.5", label="ideal",
    )
    pressure = equations_of_state.van_der_waals_pressure(
        n_particles, temperature, volumes, a, b
    )
    ax.plot(volumes * 1e3, pressure / 1e3, color="#2563eb", label="van der Waals")
    ax.set_xlabel("V (L)")
    ax.set_ylabel("P (kPa)")
    ax.set_title(f"N = {n_particles:.2e}, T = {temperature:.0f} K")
    ax.legend()
    plt.tight_layout()
    plt.show()


widgets.interact_manual(
    explore,
    n_e22=widgets.FloatSlider(min=1.0, max=20.0, step=0.5, value=5.0, description="N (1e22)"),
    temperature=widgets.FloatSlider(
        min=100.0, max=500.0, step=10.0, value=300.0, description="T (K)"
    ),
    a_fraction=widgets.FloatSlider(
        min=0.0, max=3.0, step=0.1, value=1.0, description="a / argon a"
    ),
    b_fraction=widgets.FloatSlider(
        min=0.0, max=3.0, step=0.1, value=1.0, description="b / argon b"
    ),
);

## Check your understanding

Run the cell below for the auto-graded quiz. The same questions, with written explanations
for every option, are on the module page.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "02-equations-of-state.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## Before you leave

Write a few sentences on each, in the cell below.

1. A student says: "Real gases deviate from the ideal gas law only because their molecules
   take up space." Improve this sentence so that it is actually correct, and say precisely
   what it leaves out.
2. Explain, without equations, why doubling both the amount of a real gas and its
   container's volume, at the same temperature, leaves its pressure unchanged.
3. Name one thing today's numerical agreement between the van der Waals model and argon's
   measured critical point does *not* establish.

**Your answers:**

1.
2.
3.